# POPSIGN — Landmark Extraction Driver

**What this notebook does.** Drives MediaPipe Holistic landmark extraction over
the raw POPSIGN videos (~870GB, ~64K train+test clips) using
`modules/dataset/landmark/extraction.py`. **Pipeline stage 0 for POPSIGN** —
everything downstream (feature caches, training) consumes its npz output.
Replaces the deleted `popsign.0.dataset.ipynb` stub; `popsign.1.mediapipe.ipynb`
is stale and slated for retirement (TODO §0.1).

Extraction runs **here, in this notebook** — the pilot and both bulk runs call
`extract_dataset` directly.

## Two things that had to be fixed to make that work (2026-07-19)

Both were real, and both are worth knowing about because they fail *silently*:

**1. A leaked MediaPipe graph would deadlock the pool.** POPSIGN mixes
resolutions (1944×2592 and 1080×1920). The graph's segmentation-smoothing
calculator compares each frame with the previous one, so reusing a landmarker
across a resolution change raises `RET_CHECK ... current_mat->rows ==
previous_mat->rows` — and the old retry path rebuilt the landmarker *without
closing the old one*, leaking a native graph and ~70 threads each time. A worker
reached 219 threads / 1.3 GB, then stopped at 0% CPU; `imap_unordered` cannot
tell a live-but-hung worker from a slow one, so the run hung at 18/20 with no
error. Fixed by closing before rebuilding, rebuilding *proactively* on a
resolution change, and `maxtasksperchild=64` as a safety net.

**2. Worker logs would have flooded this notebook.** MediaPipe logs from C++
(absl/glog) straight to file descriptor 2, and ipykernel captures fd-level
output into cell output by default (`IPKernelApp.capture_fd_output`). Left
alone, a 30K-video run would write hundreds of `oneDNN`/`XNNPACK`/
`inference_feedback_manager` lines *per worker* into the `.ipynb` — how this
repo once ended up with a 17 MB notebook (TODO §0.2). `extract_dataset` now
redirects each worker's fd 2 to **`<out_dir>/<split>/_worker_stderr.log`**, so
the messages stay readable on disk and out of the notebook.

Verified in a real ipykernel over ZMQ: 4 videos / 2 workers through the exact
resolution sequence that used to deadlock — 0 failed, 21.9 s (same wall time as
a plain script), **0 noise lines and 193 characters of cell output**.

> There is also a CLI, `modules/scripts/extract_popsign.py` (`pilot` /
> `run <split>`), doing exactly what §2–§4 do. It is optional — useful only for
> an unattended run that should outlive the kernel. Both share this notebook's
> manifests and are resumable, so you can switch between them freely.

## Workflow (TODO §2)

1. **Generate + verify the video manifests**
   (`data/cache/popsign/dataframes/{train,test}.csv`).
2. **Pilot batch — max 100 videos**: sweep worker counts under the resource cap,
   measure videos/s + frames/s, project full-run ETA and disk usage. **Run this
   before any bulk extraction** and pick `BEST_N_WORKERS` from its table. Pilot
   npz output is throwaway → it goes to `data/temp/popsign_pilot/` and is
   deleted by the §2 cleanup cell once `eta.json` is recorded.
3. **Full extraction** for `train` and `test` — resumable, interruption-safe.
4. **QC** — manifest stats, failure reasons, npz spot-checks.

## Confidence thresholds (TODO §2.3)

`CONFIDENCE` in the setup cell holds the detector thresholds the runs use — an
arm of the sweep in `popsign.0.dataset.confidence-tuning.ipynb`, recorded into
each run's summary so the output says what produced it. Findings so far:
`docs/reports/confidence-tuning.md` — the *pose* thresholds gate the hands
(`min_hand_landmarks_confidence` is inert), and the measured spread across arms
is small, so MediaPipe's defaults (`{}`) are the interim choice.

## Output

One npz per video at `<root>/data/raw/popsign/<split>/<label>/<id>.npz`, where
`<root>` is `POPSIGN_LANDMARKS_DRIVE` from `.env` when set (else the gitignored
`src/data/raw/` tree — fine for landmarks, they are ~3 orders of magnitude
smaller than the videos). Extracted landmarks are **persistent** data, hence
`data/raw/`; only the pilot output is temp.

| key | content |
|---|---|
| `landmarks` | `(T, 543, 3)` float16, **GISLR holistic row order** (face 0-467, left_hand 468-488, pose 489-521, right_hand 522-542), NaN where undetected — `modules.dataset.landmark.subsets` indices apply unchanged |
| `fps` / `num_frames` | video metadata |

## Artifacts of this notebook (under `data/cache/popsign/extraction/`)

| path | content |
|---|---|
| `pilot_sample.json` | the seeded ≤100-video pilot sample (stable across re-runs) |
| `pilot_results.csv` | videos/s, frames/s, CPU% per worker count |
| `pilot_throughput.png` | throughput + CPU vs worker count |
| `eta.json` | chosen worker count + projected full-run wall time and disk usage |
| `{train,test}_summary.json` | the summary of each bulk run |
| `<out_dir>/<split>/_manifest.json` | per-video extraction state (next to the npz output) |
| `<out_dir>/<split>/_worker_stderr.log` | MediaPipe's C++ logs, kept out of the notebook |

## Resumability

The extraction module follows the repo manifest pattern (TODO §1.2): npz written
atomically (temp + `os.replace`) **before** the video is marked `done`; `done`
skipped, `failed` retried on re-run; manifest saves are atomic and batched
(every 50 videos — per-video rewrites of a ~30K-entry JSON would dominate
runtime; an interrupt therefore redoes at most the last <50 videos, whose
existing npz files are then *adopted* into the manifest, not re-extracted).

Because the manifest is the state, **interrupting the kernel is safe** and
re-running the cell resumes where it stopped. Nothing depends on notebook
variables surviving.

## Resource cap (the ≤70% rule)

- **CPU**: default worker count = `floor(0.70 × logical cores)`; each worker is
  pinned to single-threaded math libs. The pilot sweep verifies measured CPU%.
- **RAM**: the job feeder blocks while system RAM ≥ 70% (psutil backpressure).
- **GPU**: untouched — the MediaPipe GPU delegate is Ubuntu-only (README
  constraints), so extraction is CPU-only by construction.

## Design decisions vs TODO §2

- Extraction logic lives in the importable module, **not in a cell**. This is a
  hard requirement, not style: Windows `spawn` pickles worker functions by
  qualified name, so a notebook-defined worker genuinely cannot be reconstructed
  in the child. Module-level workers are reconstructed fine — which is why
  calling `extract_dataset` from a cell works.
- `modules.paths.resolve_datasets()` is called only by §1, and only when the
  manifests are missing — see that section's guard.
- Only 1 of 4 POPSIGN train datasets is currently downloaded/enabled — the
  manifests cover that subset; regenerating with all four is TODO §2.2.

In [ ]:
# ============================================================
# Imports & config
# ============================================================
import json
import shutil
from multiprocessing import cpu_count
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import modules.dataset.landmark.extraction as ex
from modules.paths import CACHE_DIR, TEMP_DIR, cleanup_temp, resolve_datasets

SEED = 42  # pilot sampling seed
TRAIN_CSV = CACHE_DIR / "popsign" / "dataframes" / "train.csv"
TEST_CSV = CACHE_DIR / "popsign" / "dataframes" / "test.csv"
NB_CACHE_DIR = CACHE_DIR / "popsign" / "extraction"
NB_CACHE_DIR.mkdir(parents=True, exist_ok=True)

PILOT_MAX_VIDEOS = 100  # hard cap on the pilot batch
WORKER_COUNTS = (6, 10, 14, 19)  # sweep; 19 = floor(0.70 × 28 logical cores)
VIDEOS_PER_TRIAL = 20  # per worker count (4 × 20 = 80 ≤ 100)
CPU_FRACTION = 0.70  # resource cap
RAM_LIMIT_PCT = 70.0

# Detector thresholds — an arm of the confidence sweep
# (confidence_tuning/configs.json), so the output records what produced it.
# Interim choice per docs/reports/confidence-tuning.md; {} = MediaPipe defaults.
CONFIDENCE = {}

OUT_ROOT = ex.landmarks_root()  # resolved from POPSIGN_LANDMARKS_DRIVE / fallback
PILOT_ROOT = TEMP_DIR / "popsign_pilot"  # throwaway → data/temp, cleaned in §2


def split_progress(split: str) -> dict:
    """Extraction state for a split, read off the manifest on disk.

    Safe to call at any time, including from a second kernel while a run is in
    flight: the manifest is written atomically and only ever runs behind
    reality, never ahead. Never raises — a missing manifest or video CSV is a
    normal "not started yet" state.
    """
    csv = TRAIN_CSV if split == "train" else TEST_CSV
    total = len(pd.read_csv(csv)) if csv.exists() else None
    statuses = pd.Series(
        {k: v["status"] for k, v in ex.load_manifest(OUT_ROOT / split).items()},
        dtype=object,
    )
    done = int((statuses == "done").sum()) if len(statuses) else 0
    return {
        "split": split,
        "total": total if total is not None else "no manifest csv",
        "done": done,
        "failed": int((statuses == "failed").sum()) if len(statuses) else 0,
        "pct": round(100 * done / total, 1) if total else None,
    }


print(f"logical cores : {cpu_count()}  → default workers {ex.default_n_workers(CPU_FRACTION)}")
print(f"output root   : {OUT_ROOT.resolve()}")
print(f"pilot root    : {PILOT_ROOT} (temp — deleted after the pilot)")
print(f"model         : {ex.DEFAULT_MODEL_PATH}  (exists: {ex.DEFAULT_MODEL_PATH.exists()})")
print(f"nb cache      : {NB_CACHE_DIR}")
print(f"confidence    : {CONFIDENCE or 'MediaPipe defaults'}")
print(f"in jupyter    : {ex.in_jupyter()}  → the worker pool runs right here")

## 1. Video manifests — generate & verify (TODO §0.4/§2.2)

The manifests (`file_path, label, id`, one row per video) are what every later
stage consumes — the extraction CLI reads them, the pilot samples from them, and
the ETA is projected from their row counts. The pre-restructure copies were
cleared on 2026-07-18, so the **first cell regenerates them** by walking the raw
video tree; it is guarded by `FORCE_REGENERATE` and skips entirely when both CSVs
already exist, so it is safe to run top-to-bottom (that guard is also what keeps
a plain re-run from ever calling `resolve_datasets()`).

Both splits are laid out as `<root>/<sign>/<sign>/<id>.mp4`, so the **label** is
the parent directory and the **id** is the file stem — the same id the extraction
manifest and the confidence-tuning sample key on. The generator asserts id
uniqueness and cross-checks each label against the sign encoded in the filename,
so a change to the directory layout fails loudly instead of producing a manifest
with silently wrong labels.

**Coverage today: 30,867 train videos across 72 labels, 33,600 test videos across
250 labels.** The asymmetry is not a bug — three of the four POPSIGN train
datasets (~650GB) are still commented out in `modules/paths.py`, so 178 of the
250 test labels currently have no train videos at all. Extracting test in full is
still the right move (landmarks are cheap and it never has to be redone); those
classes just aren't trainable yet. Re-running the generator after enabling the
remaining parts extends the train manifest, and the bulk extraction picks up the
difference incrementally.

The second cell verifies the manifests still point at real files (kagglehub
version directories can shift) and summarizes them.

In [12]:
# ============================================================
# Regenerate the video manifests from the raw video tree
# -> data/cache/popsign/dataframes/{train,test}.csv  (TODO §0.4/§2.2)
# ============================================================
FORCE_REGENERATE = False   # True → rebuild even if the CSVs already exist

DATAFRAMES_DIR = CACHE_DIR / "popsign" / "dataframes"
DATAFRAMES_DIR.mkdir(parents=True, exist_ok=True)
_targets = {"train": TRAIN_CSV, "test": TEST_CSV}

if not FORCE_REGENERATE and all(p.exists() for p in _targets.values()):
    print("manifests already exist — set FORCE_REGENERATE = True to rebuild")
    for _split, _csv in _targets.items():
        print(f"  {_split}: {_csv}")
else:
    # resolve_datasets() downloads every *enabled* dataset. Only 1 of 4 POPSIGN
    # train parts is enabled in modules/paths.py (the other three are ~650GB and
    # stay commented out), so for already-downloaded data this is a cache hit —
    # but it is the one call in this notebook that *can* pull data, which is why
    # the guard above exists: a top-to-bottom run never reaches it once the CSVs
    # are in place.
    _ds = resolve_datasets()
    ROOTS = {"train": list(_ds["TRAIN"]), "test": [_ds["TEST"]]}
    for _split, _roots in ROOTS.items():
        print(f"{_split} root(s): {[r.as_posix() for r in _roots]}")

    # Layout is <root>/<sign>/<sign>/<id>.mp4 in BOTH splits, so the label is the
    # parent directory and the id is the file stem — the same id the extraction
    # manifest and the confidence-tuning sample key on.
    for _split, _roots in ROOTS.items():
        rows = [{"file_path": p.as_posix(), "label": p.parts[-2], "id": p.stem}
                for root in _roots for p in root.rglob("*.mp4")]
        df = pd.DataFrame(rows).sort_values("id", ignore_index=True)
        assert len(df), f"{_split}: no .mp4 files found under {_roots}"
        assert df["id"].is_unique, f"{_split}: duplicate video ids"
        # the sign is encoded in the filename too, so a disagreement means the
        # directory layout changed and parts[-2] is no longer the label
        _bad = df[~df.apply(lambda r: f"-{r.label}-" in r.id, axis=1)]
        assert _bad.empty, (f"{_split}: {len(_bad)} ids disagree with their folder "
                            f"label, e.g. {_bad.head(3)['id'].tolist()}")
        df.to_csv(_targets[_split], index=False)
        print(f"  wrote {_targets[_split].name}: {len(df):,} videos · "
              f"{df['label'].nunique()} labels")

    # Coverage note: TEST ships all 250 signs, but TRAIN only has the labels in
    # the enabled part(s). Extracting test in full is still correct — it just
    # cannot all be trained on until the remaining train parts are downloaded.
    _tr, _te = pd.read_csv(TRAIN_CSV), pd.read_csv(TEST_CSV)
    _only_test = set(_te["label"]) - set(_tr["label"])
    if _only_test:
        print(f"\nNOTE: {len(_only_test)} of {_te['label'].nunique()} test labels have "
              f"NO train videos (only {_tr['label'].nunique()} train labels available)"
              f"\n      → 3 of 4 POPSIGN train parts are still commented out in "
              f"modules/paths.py (TODO §2.2)")

manifests already exist — set FORCE_REGENERATE = True to rebuild
  train: C:\Users\user2\sign2speech\src\data\cache\popsign\dataframes\train.csv
  test: C:\Users\user2\sign2speech\src\data\cache\popsign\dataframes\test.csv


In [13]:
# ============================================================
# Load manifests + spot-check that the video files exist
# ============================================================
N_SPOTCHECK = 25  # random paths verified per split



frames = {}
for split, csv in [("train", TRAIN_CSV), ("test", TEST_CSV)]:
    df = pd.read_csv(csv)
    assert {"file_path", "label", "id"} <= set(df.columns), f"{csv}: unexpected schema"
    assert df["id"].is_unique, f"{csv}: duplicate video ids"
    sample = df.sample(min(N_SPOTCHECK, len(df)), random_state=SEED)["file_path"]
    missing = [p for p in sample if not Path(p).exists()]
    frames[split] = df
    print(
        f"{split}: {len(df):,} videos · {df['label'].nunique()} labels · "
        f"spot-check missing {len(missing)}/{len(sample)}"
    )
    if missing:
        print("  e.g.", missing[:3])

train_df, test_df = frames["train"], frames["test"]
train_df.groupby("label").size().describe().round(1)

train: 30,867 videos · 72 labels · spot-check missing 0/25
test: 33,600 videos · 250 labels · spot-check missing 0/25


count     72.0
mean     428.7
std      122.7
min      147.0
25%      318.8
50%      462.5
75%      539.0
max      571.0
dtype: float64

## 2. Pilot batch — parameter optimization + speed (max 100 videos)

Sweeps `WORKER_COUNTS` on **disjoint** slices of a seeded 100-video sample
(disjoint so no trial is sped up by another's cached output; sample recorded to
`pilot_sample.json` so re-runs measure the same videos). Each trial reports
videos/s, frames/s and measured CPU% — pick the count with the best throughput
whose CPU stays under the cap.

Pilot npz files are **throwaway** and go to the temp tree
`data/temp/popsign_pilot/w<N>/`; the cleanup cell below deletes them once
`eta.json` is recorded (the measurements in `data/cache/popsign/extraction/`
persist). The cell also **clears that tree before benchmarking** — a benchmark
must never resume, because `extract_dataset` skips videos already extracted, so
leftover npz from an interrupted pilot would time ~0 videos and report a
meaningless rate.

Note: the first video per worker absorbs MediaPipe graph warm-up (~seconds), so
small trials *underestimate* steady-state throughput — treat the ETA as an
upper bound.

Throughput and CPU% are plotted as two stacked panels rather than one dual-axis
chart: two y-scales on one frame make the crossing point an artifact of the
scaling choice, and the operating point is picked off *both* curves.

In [ ]:
# ============================================================
# Pilot sweep — worker counts on disjoint 20-video slices
# (pilot npz -> data/temp/popsign_pilot, deleted by the cleanup cell)
# ============================================================
_pilot_file = NB_CACHE_DIR / "pilot_sample.json"
if _pilot_file.exists():
    pilot_ids = json.loads(_pilot_file.read_text())["ids"]
else:
    pilot_ids = train_df.sample(PILOT_MAX_VIDEOS, random_state=SEED)["id"].tolist()
    _pilot_file.write_text(json.dumps(
        {"seed": SEED, "n": len(pilot_ids), "ids": pilot_ids}, indent=2))
pilot_df = train_df.set_index("id").loc[pilot_ids].reset_index()

# A benchmark must never resume: extract_dataset skips videos already done, so
# leftover npz from an earlier (or interrupted) pilot would make a trial time
# ~0 videos and report a meaningless rate. The output is throwaway by policy.
if PILOT_ROOT.exists():
    _stale = len(list(PILOT_ROOT.rglob("*.npz")))
    shutil.rmtree(PILOT_ROOT)
    print(f"cleared stale pilot output ({_stale} npz)")

print(f"pilot sample: {len(pilot_df)} videos "
      f"(using {len(WORKER_COUNTS) * VIDEOS_PER_TRIAL} across {len(WORKER_COUNTS)} trials)")

pilot_results = ex.benchmark_worker_counts(
    pilot_df, worker_counts=WORKER_COUNTS, videos_per_trial=VIDEOS_PER_TRIAL,
    pilot_root=PILOT_ROOT, cpu_fraction=CPU_FRACTION, ram_limit_pct=RAM_LIMIT_PCT,
    confidence=CONFIDENCE)
pilot_results.to_csv(NB_CACHE_DIR / "pilot_results.csv", index=False)

# Two measures on different scales: two stacked panels sharing the worker axis,
# not one chart with two y-axes.
fig, (ax_t, ax_c) = plt.subplots(2, 1, figsize=(6.4, 5), sharex=True,
                                 gridspec_kw={"height_ratios": [3, 2]})
ax_t.plot(pilot_results["n_workers"], pilot_results["videos_per_s"],
          "o-", color="#2f6fd0", lw=2, ms=8)
ax_t.set_ylabel("videos / s")
ax_t.set_title("Pilot: extraction throughput vs worker count", loc="left")
_best = pilot_results.loc[pilot_results["videos_per_s"].idxmax()]
ax_t.annotate(f"{_best['videos_per_s']:.2f} v/s @ {int(_best['n_workers'])}w",
              xy=(_best["n_workers"], _best["videos_per_s"]),
              xytext=(0, 10), textcoords="offset points",
              ha="center", fontsize=9, color="#2f6fd0", fontweight="bold")

ax_c.plot(pilot_results["n_workers"], pilot_results["cpu_mean_pct"],
          "s-", color="#d1651a", lw=2, ms=7)
ax_c.axhline(CPU_FRACTION * 100, color="#7a8794", ls="--", lw=1.2)
ax_c.annotate(f"cap {CPU_FRACTION:.0%}", xy=(0.99, CPU_FRACTION * 100),
              xycoords=("axes fraction", "data"), xytext=(0, 4),
              textcoords="offset points", ha="right", fontsize=8.5, color="#7a8794")
ax_c.set_ylabel("mean CPU %")
ax_c.set_xlabel("worker processes")
ax_c.set_ylim(0, 105)
for ax in (ax_t, ax_c):
    ax.grid(alpha=0.3)
    ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig(NB_CACHE_DIR / "pilot_throughput.png", dpi=130)
plt.show()
display(pilot_results)

In [15]:
# ============================================================
# Pick the operating point + project full-run ETA & disk usage
# ============================================================
pilot_results = pd.read_csv(NB_CACHE_DIR / "pilot_results.csv")

# best throughput among trials that respected the CPU cap (small tolerance)
_ok = pilot_results[pilot_results["cpu_mean_pct"] <= CPU_FRACTION * 100 + 5]
best = (_ok if len(_ok) else pilot_results).sort_values(
    "videos_per_s", ascending=False).iloc[0]
BEST_N_WORKERS = int(best["n_workers"])
vps = float(best["videos_per_s"])

# disk projection from the pilot npz files (temp tree — run before cleanup;
# on a re-run after cleanup the recorded eta.json mean is reused)
_pilot_npz = list(PILOT_ROOT.rglob("*.npz")) if PILOT_ROOT.exists() else []
if _pilot_npz:
    _mean_npz_mb = float(np.mean([p.stat().st_size for p in _pilot_npz]) / 1e6)
elif (NB_CACHE_DIR / "eta.json").exists():
    _mean_npz_mb = json.loads((NB_CACHE_DIR / "eta.json").read_text())["mean_npz_mb"]
else:
    _mean_npz_mb = float("nan")

eta = {"best_n_workers": BEST_N_WORKERS, "videos_per_s": vps,
       "mean_npz_mb": round(float(_mean_npz_mb), 3)}
for split, df in [("train", pd.read_csv(TRAIN_CSV)), ("test", pd.read_csv(TEST_CSV))]:
    manifest = ex.load_manifest(OUT_ROOT / split)
    pending = len(ex.pending_jobs(df, OUT_ROOT / split, dict(manifest)))
    eta[split] = {"videos": len(df), "pending": pending,
                  "eta_h": round(pending / vps / 3600, 1),
                  "projected_gb": round(len(df) * _mean_npz_mb / 1e3, 1)}
(NB_CACHE_DIR / "eta.json").write_text(json.dumps(eta, indent=2))
print(json.dumps(eta, indent=2))

FileNotFoundError: [Errno 2] No such file or directory: 'C:\\Users\\user2\\sign2speech\\src\\data\\cache\\popsign\\extraction\\pilot_results.csv'

In [ ]:
# ============================================================
# Pilot cleanup — the pilot npz output is throwaway (data/temp policy):
# delete the whole temp tree now that eta.json + pilot_results.csv are
# recorded in data/cache/popsign/extraction/. Safe to re-run.
# ============================================================
assert (NB_CACHE_DIR / "eta.json").exists(), \
    "run the ETA cell first — cleanup would discard the unmeasured pilot output"
cleanup_temp()
print(f"deleted temp tree ({TEMP_DIR}) — pilot measurements persist in {NB_CACHE_DIR}")

## 3. Full extraction — train

Resumable: **interrupt the kernel any time** — npz writes are atomic and the
manifest only ever runs behind reality, never ahead — and re-running this cell
skips everything already done. `LIMIT` allows staged runs (e.g. `LIMIT = 2000`
tonight, `None` for the rest). One progress bar shows live CPU/RAM and the
failure count; expect the `eta.json` wall time at `BEST_N_WORKERS` from §2.

MediaPipe's C++ logging goes to `<out_dir>/train/_worker_stderr.log`, not into
this cell — see the title cell for why that matters.

The cell prints manifest state before and after, so re-running it after an
interrupt tells you exactly how much is left. `split_progress("train")` is also
safe to call from a *second* kernel while this one is busy.

In [ ]:
# ============================================================
# FULL RUN — train split (resumable; interrupt-safe)
# ============================================================
N_WORKERS = int(json.loads((NB_CACHE_DIR / "eta.json").read_text())["best_n_workers"])
LIMIT = None    # int → process only that many pending videos this run

print("before:", split_progress("train"))

train_summary = ex.extract_dataset(
    pd.read_csv(TRAIN_CSV), split="train", out_root=OUT_ROOT,
    n_workers=N_WORKERS, cpu_fraction=CPU_FRACTION, ram_limit_pct=RAM_LIMIT_PCT,
    limit=LIMIT, confidence=CONFIDENCE)

(NB_CACHE_DIR / "train_summary.json").write_text(json.dumps(train_summary, indent=2))
print(json.dumps(train_summary, indent=2))
print("after :", split_progress("train"))
print(f"\nworker logs: {OUT_ROOT / 'train' / '_worker_stderr.log'}")

## 4. Full extraction — test

Identical to §3 for the test split. Note that test carries **all 250 labels**
while train currently has 72 (§1), so this is the larger run of the two — 33,600
videos against 30,867.

In [ ]:
# ============================================================
# FULL RUN — test split (resumable; interrupt-safe)
# ============================================================
N_WORKERS = int(json.loads((NB_CACHE_DIR / "eta.json").read_text())["best_n_workers"])
LIMIT = None    # int → process only that many pending videos this run

print("before:", split_progress("test"))

test_summary = ex.extract_dataset(
    pd.read_csv(TEST_CSV), split="test", out_root=OUT_ROOT,
    n_workers=N_WORKERS, cpu_fraction=CPU_FRACTION, ram_limit_pct=RAM_LIMIT_PCT,
    limit=LIMIT, confidence=CONFIDENCE)

(NB_CACHE_DIR / "test_summary.json").write_text(json.dumps(test_summary, indent=2))
print(json.dumps(test_summary, indent=2))
print("after :", split_progress("test"))
print(f"\nworker logs: {OUT_ROOT / 'test' / '_worker_stderr.log'}")

## 5. QC — manifest stats, failures, npz spot-checks

Reads only the split manifests and npz artifacts (no video access), so this is
cheap to re-run at any point mid-extraction to check progress and health.

In [ ]:
# ============================================================
# QC over both splits — progress, failures, npz spot-check
# ============================================================
N_QC_SAMPLES = 5   # npz files load-checked per split
rng = np.random.default_rng(SEED)

for split in ["train", "test"]:
    manifest = ex.load_manifest(OUT_ROOT / split)
    if not manifest:
        print(f"{split}: no manifest yet — extraction not started\n")
        continue
    st = pd.Series({k: v["status"] for k, v in manifest.items()})
    done = st[st == "done"].index
    print(f"{split}: {len(st):,} attempted — {(st == 'done').sum():,} done · "
          f"{(st == 'failed').sum():,} failed")
    fails = {k: v["error"] for k, v in manifest.items() if v["status"] == "failed"}
    if fails:
        print("  failure reasons (top):")
        print(pd.Series(list(fails.values())).value_counts().head(5).to_string())
    nf = pd.Series({k: manifest[k].get("n_frames") for k in done}).dropna()
    if len(nf):
        print(f"  frames/video: median {nf.median():.0f} · p5 {nf.quantile(.05):.0f} "
              f"· p95 {nf.quantile(.95):.0f} · zero-frame videos {(nf == 0).sum()}")
    for vid in rng.choice(done, size=min(N_QC_SAMPLES, len(done)), replace=False):
        z = np.load(manifest[vid]["artifact"])
        lm = z["landmarks"]
        assert lm.ndim == 3 and lm.shape[1:] == (543, 3), f"bad shape {lm.shape}: {vid}"
        nan_frac = float(np.isnan(lm.astype(np.float32)).mean()) if len(lm) else 1.0
        print(f"  ok {vid}: {lm.shape} float16 · NaN {nan_frac:.1%} · fps {float(z['fps']):.0f}")
    _sz = sum(Path(manifest[k]["artifact"]).stat().st_size
              for k in done if Path(manifest[k]["artifact"]).exists())
    print(f"  on disk: {_sz / 1e9:.2f} GB\n")

## Caveats & follow-ups

- **A hung worker still has no watchdog.** The known cause of the 2026-07-19
  deadlock (leaked MediaPipe graphs across a resolution change) is fixed, and
  `maxtasksperchild=64` recycles workers as a safety net — but `imap_unordered`
  fundamentally cannot distinguish a worker that is slow from one that will
  never return, so an *unknown* cause would still hang the run silently. If a
  run stops progressing: check per-worker CPU and thread count (the tell last
  time was 219 threads / 1.3 GB against 76 / 614 MB for healthy siblings).
  Tracked in TODO §2.1.
- **Manifests currently cover 1 of 4 POPSIGN train datasets** — the other three
  (~650GB) are commented out in `modules/paths.py` (`resolve_datasets()`). So
  train has 72 labels while test has all 250, and **178 test labels have no
  train videos**. Extracting test in full is still right (landmarks are cheap
  and never need redoing); those classes just aren't trainable yet. Enable +
  download the parts, re-run §1 with `FORCE_REGENERATE = True`, then re-run §3 —
  resumability makes the union incremental. Tracked in TODO §2.2.
- **`CONFIDENCE` is an interim choice.** The confidence sweep
  (`docs/reports/confidence-tuning.md`) has 2 of 7 arms measured, and its quality
  proxies are contaminated by clip padding — ~half of every POPSIGN clip is
  lead-in/lead-out with no hands in it. MediaPipe's defaults lead the paired
  comparison that exists, and the spread across arms is ~0.02 of hand-detection
  rate, so this is unlikely to be the decision that matters. Re-check before
  committing the full ~64K-video run.
- The pilot ETA is an **upper bound** (warm-up bias) but assumes the machine is
  otherwise idle; heavy concurrent use (e.g. GPU training with data loading)
  will stretch it.
- Pilot output lives in `data/temp/popsign_pilot/` and is deleted by the §2
  cleanup cell — per the repo temp policy, nothing under `data/temp/` survives
  the notebook that wrote it.
- `_worker_stderr.log` sits **next to the npz output**, not in the repo, and is
  append-only across runs — check it after a run with failures, and delete it
  freely (nothing reads it back).
- `fps` is taken from the container metadata (`cv2.CAP_PROP_FPS`, fallback 30
  when the container reports 0) and is stored per npz; downstream consumers
  should not assume a uniform frame rate across POPSIGN.
- Landmark values are float16 (halves storage; x/y are normalized [0,1] where
  ~3 significant digits survive, which is far above landmark jitter).